In [ ]:
import cv2, dlib, librosa, numpy as np
import pandas as pd
from xgboost import XGBClassifier
import joblib
import os

# 🔧 Load models
face_model = XGBClassifier()
face_model.load_model("xgb_face_model.json")
stack_model = joblib.load("stacked_model.pkl")

# 📍 Dlib detectors
face_detector = dlib.get_frontal_face_detector()
landmark_predictor = dlib.shape_predictor("shape_predictor_68_face_landmarks.dat")

# 🎞️ Input paths (change these)
video_path = "test_clip.mp4"
audio_path = "test_clip.wav"
clip_name = os.path.basename(video_path)

# 🎥 Extract facial features
cap = cv2.VideoCapture(video_path)
landmarks = []
frame_count = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    if frame_count % 30 == 0:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = face_detector(gray)
        if faces:
            shape = landmark_predictor(gray, faces[0])
            vec = np.array([[pt.x, pt.y] for pt in shape.parts()]).flatten()
            landmarks.append(vec)
    frame_count += 1
cap.release()

# 🧠 Predict face score
avg_landmarks = np.mean(landmarks, axis=0)
face_score = face_model.predict_proba(avg_landmarks.reshape(1, -1))[0][1]

# 🔊 Extract MFCCs
y, sr = librosa.load(audio_path, sr=None)
mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
mfcc_vec = np.mean(mfcc, axis=1)

# 🧱 Build fusion input
fusion_input = pd.DataFrame({
    "face_score": [face_score],
    **{f"mfcc_{i}": [val] for i, val in enumerate(mfcc_vec)}
})

# 🌀 Predict final label
pred = stack_model.predict(fusion_input)[0]
conf = stack_model.predict_proba(fusion_input)[0][1]
label = "Truthful" if pred == 1 else "Deceptive"

# 📣 Output
print(f"🎞️ {clip_name} → {label}  (Confidence: {conf:.2f})")
